In [ ]:
from pathlib import Path
import pandas as pd
import tensorflow as tf ## deep learning library which will handle loading image tensors, building nn, training the modelm calculating loss, updating weights and making predicitions
import matplotlib.pyplot as plt

2.21.0


load split csv files

In [3]:
PROJECT_ROOT = Path("..")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")

train_df.head()

,original_path,processed_path,label
0,..\data\raw\inflammatory_acne\inflam_021.jpg,..\data\processed\inflammatory_acne\inflammato...,inflammatory_acne
1,..\data\raw\comedonal_closed\come_048.png,..\data\processed\comedonal_closed\comedonal_c...,comedonal_closed
2,..\data\raw\comedonal_open\comeo_008.png,..\data\processed\comedonal_open\comedonal_ope...,comedonal_open
3,..\data\raw\acne_scars\scar_016.png,..\data\processed\acne_scars\acne_scars_0016.jpg,acne_scars
4,..\data\raw\acne_scars\scar_041.png,..\data\processed\acne_scars\acne_scars_0041.jpg,acne_scars


get class names

In [4]:
class_names = sorted(train_df["label"].unique())
class_names

['acne_excoriee',
 'acne_scars',
 'comedonal_closed',
 'comedonal_open',
 'inflammatory_acne']

create a label mapping
ie: 0 = class 1 label, 1 = class 2, 2 = class 3, etc

need this since ml models work with numbers instead of text process = label encoding

In [5]:
label_to_index = {label: index for index, label in enumerate(class_names)}
label_to_index

{'acne_excoriee': 0,
 'acne_scars': 1,
 'comedonal_closed': 2,
 'comedonal_open': 3,
 'inflammatory_acne': 4}

add the numeric labels to dataframes df

In [6]:
train_df["label_index"] = train_df["label"].map(label_to_index)
val_df["label_index"] = val_df["label"].map(label_to_index)
test_df["label_index"] = test_df["label"].map(label_to_index)

train_df.head()

,original_path,processed_path,label,label_index
0,..\data\raw\inflammatory_acne\inflam_021.jpg,..\data\processed\inflammatory_acne\inflammato...,inflammatory_acne,4
1,..\data\raw\comedonal_closed\come_048.png,..\data\processed\comedonal_closed\comedonal_c...,comedonal_closed,2
2,..\data\raw\comedonal_open\comeo_008.png,..\data\processed\comedonal_open\comedonal_ope...,comedonal_open,3
3,..\data\raw\acne_scars\scar_016.png,..\data\processed\acne_scars\acne_scars_0016.jpg,acne_scars,1
4,..\data\raw\acne_scars\scar_041.png,..\data\processed\acne_scars\acne_scars_0041.jpg,acne_scars,1


ml overview:

in this case the image is the input x and the label is the target y

each row is basically x and y where its image path and numeric class label

In [9]:
train_paths = train_df["processed_path"].values
val_paths = val_df["processed_path"].values
test_paths = test_df["processed_path"].values

train_labels = train_df["label_index"].values
val_labels = val_df["label_index"].values
test_labels = test_df["label_index"].values

image loading function:

essentially tells tf how to turn an image file path into an actual model input

In [8]:
IMG_SIZE = 224

def load_image(image_path, label):
    image = tf.io.read_file(image_path) ## read the file 
    image = tf.image.decode_jpeg(image, channels=3) ## decode the jpeg into a numerical image tensor
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE)) ## resize, safety check, since nn needs consistent input size
    image = image / 255.0 ## normalize to a scale of 0 to 1 instead of 0 to 225 since deep learning models can train better on inputs that are smaller
    
    return image, label

tensorflow datasets:

essentially a way of efficiently handling data, lets tf load images in a structure pipeline

In [10]:
BATCH_SIZE = 16

train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
val_ds = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
test_ds = tf.data.Dataset.from_tensor_slices((test_paths, test_labels))

In [11]:
## apply image loading function, basically applies load_image to every item in the dataset
## it chanes image path, label to actual image tensor, label

## before: "../data/processed/acne_scars/acne_scars_0001.jpg", 1
## after: image_array_with_shape_(224,224,3), 1

train_ds = train_ds.map(load_image)
val_ds = val_ds.map(load_image)
test_ds = test_ds.map(load_image)

need to shuffle and batch:

shuffling randomizes the order of training images, this is needed since images are ordered by class and the model could learn in a biased order, so shuffling will give the model a mixed set of examples 

we also only shuffle the training dataset the model learns from it

validation and test data dont need to since we evaluate on them

batching gives the model a group of images instead of one at a time, this done since training one image at a time is slow and noisy and can be memory heavy if training the whole dataset at once 

In [12]:
train_ds = train_ds.shuffle(buffer_size=len(train_df)).batch(BATCH_SIZE)
val_ds = val_ds.batch(BATCH_SIZE)
test_ds = test_ds.batch(BATCH_SIZE)

cnn = convolutional neural network and are desined for images 

the baseline model which used logistic regression flatted images immediately 
224 x 224 x 3 = 150528 pixle features which destroys spatial structure

a cnn keeps the image shape and learns local patterns 

in this case for example it can learn (hopefully it can):
small red regions, clusters of bumps, skin texture changes, edges, dark sports and scar texture

core ideas:

convultional layers, are thee core building block of a cnn, it is what allows the network to detect visual patterns in an image

instead of looking at the whole image at oncem a CL scans the image with a small grid of numbers called a filter or kernel, it slides across the image patch by patch and doing simple multiplication and sum at each position, the result is a map of where that pattern appears int he image 


In [13]:
num_classes = len(class_names)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224, 224, 3)), ## each image will have the shape of 224 224 3
    ## first convolution layer, learns 32 small filers, each of size 3 x 3
    ## relu = rectified linear unit helps model learn non-linear patterns
    tf.keras.layers.Conv2D(32, (3, 3), activation="relu"), 
    tf.keras.layers.MaxPooling2D(), ## reduces image size while keeping important features, makes training faster, reduces memory use, keeps strongest detected patterns, help model focus on important features 

    ## second confultion layer, learns 64 filiters
    ## as the model goes deeper it leanrs more complex patterns
    tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    ## third convolution layer, learns 128 filters
    tf.keras.layers.Conv2D(128, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(), ## used to turn the visual feature maps into a vector so that the final classification layers can use them

    ## baseline flattened the raw image immediately, cnn flattens only after learning spatial features 

    ## dense layer is a fully connected layer, where it takes the learned image features and starts combining them to make a classification decision
    tf.keras.layers.Dense(128, activation="relu"), 
    tf.keras.layers.Dropout(0.3), ## used to reduce overfitting,0.3 = ignore 30 of this layers neurons

    ## overfitting happens when the model meorizes the training images instead of learn the general patterns

    ## ie: traing accuracy = 90%, validation accuracy = 35% means the model is doing good on images it has seen but bad on new images 

    tf.keras.layers.Dense(num_classes, activation="softmax")
])

In [ ]:
## compile model:

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)